# TeraiNet inference example

Follow [mewc-predict](https://github.com/zaandahl/mewc-predict/blob/main/src/mewc_predict.py) for the general procedure

## Setup

### Imports

Follow [mewc-flow](https://github.com/zaandahl/mewc-flow/blob/main/requirements.txt) for the key package versions

In [1]:
!pip install keras==3.3.3 kimm==0.2.5 tensorflow==2.16.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 3.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.5.0
    Uninstalling ml_dtypes-0.5.0:
      Successfully uninstalled ml_dtypes-0.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.4.35 requires ml-dtypes>=0.4.0, but you have ml-dtypes 0.3.2 which is incompatible.


In [2]:
import os
import yaml
import random
import shutil
import pandas as pd
from pathlib import Path

import kimm
import tensorflow as tf
from keras import saving

### Utilities

In [3]:
def copy_random_images(src_dir, dst_dir, n, seed=42):
    """
    Copies n random images from src_dir to dst_dir.

    Args:
        src_dir (str): Source directory containing images.
        dst_dir (str): Destination directory to copy images into.
        n (int): Number of images to copy.
        seed (int): Random seed for reproducibility.
    """
    # Ensure target directory exists
    os.makedirs(dst_dir, exist_ok=True)

    # List all files in the source directory
    all_files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]

    # Check if n is greater than available files
    if n > len(all_files):
        raise ValueError(f"Requested {n} images, but only {len(all_files)} available in source directory.")

    # Randomly sample n files
    random.seed(seed)
    selected_files = random.sample(all_files, n)

    # Copy each file to the target directory
    for filename in selected_files:
        src_path = os.path.join(src_dir, filename)
        dst_path = os.path.join(dst_dir, filename)
        shutil.copy2(src_path, dst_path)

    print(f"Copied {n} random images from '{src_dir}' to '{dst_dir}'.")

## Prepare images

In [4]:
copy_random_images("../input/preprocess-images/terainet_images/test2/class_1", "../images", 10)

Copied 10 random images from '../input/preprocess-images/terainet_images/test2/class_1' to '../images'.


In [5]:
img_generator = tf.keras.preprocessing.image_dataset_from_directory(
    "../images", 
    labels=None,
    label_mode=None,
    batch_size=8, 
    image_size=(224, 224),
    shuffle=False
)

Found 10 files.


## Predict

In [6]:
model = saving.load_model("../input/train-and-evaluate-terainet/model.keras", compile=False)

In [7]:
preds = model.predict(img_generator)

2/2 ━━━━━━━━━━━━━━━━━━━━ 14s 7s/step


In [8]:
preds

array([[5.2381847e-02, 1.9996221e-01, 2.8495291e-01, 4.6548069e-02,
        1.1974893e-01, 1.3221885e-01, 1.8170491e-02, 5.3616375e-02,
        5.2950364e-02, 3.9450005e-02],
       [2.5736994e-01, 1.0949787e-01, 1.6122012e-01, 1.3590532e-02,
        8.4038295e-02, 6.8595107e-03, 8.9867301e-03, 7.5156413e-02,
        6.8466127e-02, 2.1481442e-01],
       [1.9459581e-02, 1.6660877e-02, 9.0841949e-02, 1.2797284e-01,
        1.3810986e-01, 3.7299789e-02, 7.0161916e-02, 1.2626718e-01,
        3.3128676e-01, 4.1939180e-02],
       [7.7461773e-01, 7.5880433e-03, 8.1880882e-02, 4.1598991e-02,
        9.0330606e-03, 1.7620470e-02, 2.0352520e-02, 1.9009238e-02,
        8.6090667e-03, 1.9689834e-02],
       [8.7063599e-01, 5.8444124e-03, 5.6552164e-02, 1.0452784e-02,
        1.8009124e-02, 5.6710229e-03, 5.4316991e-03, 1.3848719e-02,
        6.1928863e-03, 7.3611112e-03],
       [8.0514151e-01, 1.4395743e-02, 8.6553395e-02, 2.4525946e-02,
        3.9340876e-02, 7.6304758e-03, 5.0827600e-03, 1.06

## Post-processing

Follow [mewc-predict](https://github.com/zaandahl/mewc-predict/blob/main/src/mewc_predict.py)

In [9]:
with open("../input/train-and-evaluate-terainet/class_list.yaml", "r") as file:
    class_map = yaml.safe_load(file)
class_map

{'1': 'tiger',
 '10': 'bird',
 '2': 'leopard',
 '3': 'black_bear',
 '4': 'other_carnivores',
 '5': 'deer',
 '6': 'wild_boar',
 '7': 'buffalo',
 '8': 'rhino',
 '9': 'elephant'}

In [10]:
inv_class = {v: k for k, v in class_map.items()}
inv_class

{'tiger': '1',
 'bird': '10',
 'leopard': '2',
 'black_bear': '3',
 'other_carnivores': '4',
 'deer': '5',
 'wild_boar': '6',
 'buffalo': '7',
 'rhino': '8',
 'elephant': '9'}

In [11]:
file_paths = img_generator.file_paths
filenames = list(map(lambda x : Path(x).name, file_paths))
labels = list(map(lambda x : Path(x).parent.name, file_paths))

In [12]:
class_ids = sorted(inv_class.values())
class_names = [class_map.get(i,i)  for i in class_ids]
pred_df = pd.DataFrame(preds, columns=class_ids)
pred_df.head()

,1,10,2,3,4,5,6,7,8,9
0,0.052382,0.199962,0.284953,0.046548,0.119749,0.132219,0.018170,0.053616,0.052950,0.039450
1,0.257370,0.109498,0.161220,0.013591,0.084038,0.006860,0.008987,0.075156,0.068466,0.214814
2,0.019460,0.016661,0.090842,0.127973,0.138110,0.037300,0.070162,0.126267,0.331287,0.041939
3,0.774618,0.007588,0.081881,0.041599,0.009033,0.017620,0.020353,0.019009,0.008609,0.019690
4,0.870636,0.005844,0.056552,0.010453,0.018009,0.005671,0.005432,0.013849,0.006193,0.007361


In [13]:
file_series = pd.Series(filenames)
label_series = pd.Series(labels)
pred_df.insert(0, "filename", file_series, True)
pred_df.insert(1, "label", label_series, True)
pred_df = pd.melt(pred_df, id_vars=['filename', 'label'], value_vars=class_ids, var_name="class_id", value_name="prob")
pred_df["class_name"] = pred_df["class_id"].replace(class_map)
pred_df["class_rank"] = pred_df.groupby("filename")["prob"].rank("average", ascending=False)
pred_df.head(10)

,filename,label,class_id,prob,class_name,class_rank
0,class_1_test2_11-0.jpg,images,1,0.052382,tiger,7.0
1,class_1_test2_145-0.jpg,images,1,0.257370,tiger,1.0
2,class_1_test2_161-0.jpg,images,1,0.019460,tiger,9.0
3,class_1_test2_180-0.jpg,images,1,0.774618,tiger,1.0
4,class_1_test2_201-0.jpg,images,1,0.870636,tiger,1.0
5,class_1_test2_22-0.jpg,images,1,0.805142,tiger,1.0
6,class_1_test2_234-0.jpg,images,1,0.761445,tiger,1.0
7,class_1_test2_305-0.jpg,images,1,0.000171,tiger,10.0
8,class_1_test2_315-0.jpg,images,1,0.881008,tiger,1.0
9,class_1_test2_318-0.jpg,images,1,0.856423,tiger,1.0


In [14]:
pred_df = pred_df[pred_df["class_rank"] == 1.0]
pred_df = pred_df.drop(["label", "class_rank"], axis=1)
pred_df

,filename,class_id,prob,class_name
1,class_1_test2_145-0.jpg,1,0.257370,tiger
3,class_1_test2_180-0.jpg,1,0.774618,tiger
4,class_1_test2_201-0.jpg,1,0.870636,tiger
5,class_1_test2_22-0.jpg,1,0.805142,tiger
6,class_1_test2_234-0.jpg,1,0.761445,tiger
8,class_1_test2_315-0.jpg,1,0.881008,tiger
9,class_1_test2_318-0.jpg,1,0.856423,tiger
20,class_1_test2_11-0.jpg,2,0.284953,leopard
47,class_1_test2_305-0.jpg,4,0.397291,other_carnivores
82,class_1_test2_161-0.jpg,8,0.331287,rhino
